# 🎬 AI Video Studio — Google Colab Launcher

This notebook clones the project, installs dependencies, mounts Google Drive,
and launches the Gradio app with a public link.

Works on **Colab Free (T4)**, **Colab Pro (L4/A100)**, and any CUDA GPU (RunPod, etc.).

**Steps:** Run all cells top to bottom (Runtime ▸ Run all).

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/your-username/AI-Video-Studio.git"  # ⬅️ replace with your fork

if not os.path.isdir("/content/AI-Video-Studio"):
    !git clone $REPO_URL /content/AI-Video-Studio
%cd /content/AI-Video-Studio

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null

## 4. Mount Google Drive and prepare the output folder

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/AI-Video-Studio"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Symlink the local outputs/ folder to Drive so every generated file is
# saved automatically, without changing any app code.
local_outputs = "/content/AI-Video-Studio/outputs"
if os.path.islink(local_outputs) or os.path.isdir(local_outputs):
    if os.path.islink(local_outputs):
        os.remove(local_outputs)
    else:
        import shutil
        shutil.rmtree(local_outputs)
os.symlink(DRIVE_FOLDER, local_outputs)
print(f"Outputs will be saved to: {DRIVE_FOLDER}")

## 5. (Optional) Choose your models
By default the app uses lightweight models that fit a free T4 (16GB).
If you have an L4/A100, you can switch to a higher-quality model.

In [ ]:
import os

# Uncomment / edit to override the defaults from config.py:
# os.environ["AIVS_T2V_MODEL"] = "wan2.1-t2v-1.3b"   # free T4-friendly
# os.environ["AIVS_T2V_MODEL"] = "cogvideox-5b"       # needs L4/A100
# os.environ["AIVS_I2V_MODEL"] = "cogvideox-5b-i2v"
print("Using defaults from config.py unless overridden above.")

## 6. Launch the app
A public `gradio.live` link will be printed below.

In [ ]:
import subprocess, sys

# Force share=True in Colab so you get a public URL.
with open("app.py") as f:
    code = f.read()
code = code.replace("share=False", "share=True")
with open("app_colab.py", "w") as f:
    f.write(code)

!python app_colab.py

---
### Troubleshooting
- **CUDA out of memory**: lower resolution/duration in the UI, or switch to a smaller model in step 5.
- **xFormers not found**: the app automatically skips it and falls back to standard attention.
- **Drive quota**: generated videos can be large; monitor your Drive storage.

See the project `README.md` for the full FAQ.